# Pair Classifier — Match vs. Ambiguous (LightGBM, **v2**)

**Entity resolution / ambiguity detection.** Over the pool of *plausible* record pairs
(a pair is kept if it is a **match** or was flagged **ambiguous**), this model predicts
whether a pair is a **confident match** (`0`) or **ambiguous / hard case** (`1`). A positive
prediction means *"route to a human reviewer"*, not *"same patient"*.

**What changed from v1 → v2: feature engineering only.** Everything else — the plausible-pairs
population, the `ambiguous_pair` target, the random stratified split, the LightGBM config — is
held identical to v1 so any metric change is attributable to features, not confounders.

v1's lesson was that ambiguity lives in **partial** agreement (a name close but not identical),
which the 3-level `same/different/missing` encoding cannot see. v2 adds:

- **Continuous string similarity** — Jaro-Winkler on name/email/address fields (best-in-class
  for short personal-name strings in record linkage), plus normalized Levenshtein on the longer
  email/address fields.
- **Street number** — first numeric token of the address, compared exactly. Matches addresses
  despite typos / abbreviations / incompleteness in the street *name*.
- **Phonetic name agreement** (Metaphone), **name-swap** detection, **birth-date deltas**,
  **SSN digit-level** agreement, **address token overlap**, and **aggregate evidence counts**.

The v1 `same/different/missing` categoricals are **kept** alongside the new features.

**Structure:** 1) setup · 2) ingestion · 3) population + target · 4) field selection ·
5) feature engineering · 6) split · 7) training · 8) evaluation · 9) v1-vs-v2 ablation.

> Standalone training/evaluation notebook — not wired into `src/pipeline.py`.

## 1. Setup & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import jellyfish

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42

In [ ]:
from pathlib import Path

# Resolve empi-service/ by walking up from the notebook until we find a data/ dir.
def _find_service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / 'data').is_dir() and (d / 'src').is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")

SERVICE_ROOT = _find_service_root(Path.cwd())
DATA_DIR = SERVICE_ROOT / 'data'

GOLD_LABELS_PATH = DATA_DIR / 'gold_labels' / 'final_gold_labels_v1_2026_07_05.csv'
CLEANED_PATH = DATA_DIR / 'processed' / 'MDM_Population_cleaned_v6_2026_06_25.parquet'

print('service root :', SERVICE_ROOT)
print('gold labels  :', GOLD_LABELS_PATH, '-> exists:', GOLD_LABELS_PATH.exists())
print('cleaned data :', CLEANED_PATH, '-> exists:', CLEANED_PATH.exists())

## 2. Data ingestion

Load the gold labels and the cleaned records, then attach A-side and B-side attributes to
every pair. Identical to v1. `dtype=str` on PATIDs preserves the leading-zeros invariant.

In [ ]:
# Gold labels: force string PATIDs (leading-zeros invariant); parse the two flags.
gold = pd.read_csv(
    GOLD_LABELS_PATH,
    dtype={'PATID_A': str, 'PATID_B': str},
)


def _to_bool(s: pd.Series) -> pd.Series:
    """Robustly parse a boolean-ish column (handles True/False, 1/0, 'true'/'false')."""
    if s.dtype == bool:
        return s
    return (
        s.astype(str).str.strip().str.lower().map(
            {'true': True, '1': True, '1.0': True,
             'false': False, '0': False, '0.0': False}
        )
    )


gold['final_gold_label'] = _to_bool(gold['final_gold_label']).fillna(False).astype(bool)
gold['ambiguous_pair'] = _to_bool(gold['ambiguous_pair']).fillna(False).astype(bool)

print('gold labels shape :', gold.shape)
print('\nfinal_gold_label x ambiguous_pair (counts):')
print(pd.crosstab(gold['final_gold_label'], gold['ambiguous_pair']))

In [ ]:
# Cleaned records: one row per PATID. Keep the attribute columns we may use as features.
ATTRIBUTE_COLS = [
    'FirstNM_clean',
    'LastNM_clean',
    'MiddleNM_clean',
    'BirthDT_clean',
    'SSN_clean',
    'last_4_SSN',
    'Email_clean',
    'ZipCD_clean_base',
    'AddressLine1_clean',
    'SexAtBirthDSC_clean',
    'Phones_set',
]

cleaned = pd.read_parquet(CLEANED_PATH)
keep = ['PATID'] + [c for c in ATTRIBUTE_COLS if c in cleaned.columns]
cleaned = cleaned[keep].copy()
cleaned['PATID'] = cleaned['PATID'].astype(str)
cleaned = cleaned.drop_duplicates(subset='PATID', keep='first').set_index('PATID')
print('cleaned records shape:', cleaned.shape)
print('columns available    :', list(cleaned.columns))

In [ ]:
# Attach A-side and B-side attributes to every pair via two indexed joins.
a = cleaned.add_suffix('_A').reindex(gold['PATID_A'].values).reset_index(drop=True)
b = cleaned.add_suffix('_B').reindex(gold['PATID_B'].values).reset_index(drop=True)

pairs = pd.concat([gold.reset_index(drop=True), a, b], axis=1)

n_unmatched = (~gold['PATID_A'].isin(cleaned.index) | ~gold['PATID_B'].isin(cleaned.index)).sum()
print(f'pairs with a PATID not found in cleaned data: {n_unmatched} / {len(pairs)}')
print('joined pairs shape:', pairs.shape)

## 3. Population filter + target definition

Identical to v1. Keep **plausible** pairs (match **or** ambiguous); drop confident non-matches.
Target `target_ambiguous`: `1` = ambiguous (route to review), `0` = confident match.

In [ ]:
keep_mask = pairs['final_gold_label'] | pairs['ambiguous_pair']
n_before = len(pairs)
pairs = pairs[keep_mask].reset_index(drop=True)
print(f'dropped {n_before - len(pairs)} confident non-match pairs; {len(pairs)} plausible pairs remain\n')

pairs['target_ambiguous'] = pairs['ambiguous_pair'].astype(int)
print('target distribution (0=confident match, 1=ambiguous):')
print(pairs['target_ambiguous'].value_counts())
print(f"\npositive (ambiguous) rate: {pairs['target_ambiguous'].mean():.3f}")

## 4. Field selection

Same field roster as v1. `SELECTED_FIELDS` drives the v1-style `same/different/missing`
categoricals; the v2 continuous/derived features (next section) draw on the same columns.

In [ ]:
AVAILABLE_FIELDS = {
    'first_name':  'FirstNM_clean',
    'last_name':   'LastNM_clean',
    'middle_name': 'MiddleNM_clean',
    'birth_date':  'BirthDT_clean',
    'email':       'Email_clean',
    'ssn':         'SSN_clean',        # full 9-digit SSN
    'address1':    'AddressLine1_clean',
    'phones':      'Phones_set',       # set-valued: compared by overlap
}
SET_FIELDS = {'phones'}

SELECTED_FIELDS = [
    'first_name', 'last_name', 'middle_name', 'birth_date',
    'email', 'ssn', 'address1', 'phones',
]
assert set(SELECTED_FIELDS) <= set(AVAILABLE_FIELDS), 'unknown field in SELECTED_FIELDS'
print('using fields:', SELECTED_FIELDS)

## 5. Feature engineering

Three groups of features. **Categorical** features keep pandas `category` dtype (LightGBM
treats them natively); **numeric** features use `float`, with `NaN` where a value is absent
(LightGBM splits on missing natively, so we do not impute).

| Group | Features |
|---|---|
| **v1 categoricals** (kept) | `cmp_<field>` — same / different / missing, for all 8 fields |
| **String similarity** | `sim_jw_*` Jaro-Winkler (first, last, middle, email, address1); `sim_lev_*` normalized Levenshtein (email, address1) |
| **Derived / structural** | `cmp_street_num`; `phon_first_match`, `phon_last_match` (Metaphone); `name_swap_score`; `dob_abs_days`, `dob_md_swap`; `ssn_digit_frac`; `addr_token_jaccard` |
| **Aggregate evidence** | `n_same`, `n_different`, `n_missing`, `n_strong_id_agree` |

**Why the street number:** the numeric token (`"123 N Main St Apt 4" → 123`) is the most stable,
most discriminative part of an address. Comparing it exactly recovers same-address pairs whose
street *name* differs by typos, abbreviations, or missing tokens — where raw string similarity
scores low.

**Why Jaro-Winkler:** it is the record-linkage standard for short personal-name strings —
bounded in [0, 1], prefix-weighted (matching how names actually vary), and transposition-aware.
Normalized Levenshtein is added on the longer email/address fields as a complementary signal.

In [ ]:
import ast, re

MISSING, SAME, DIFFERENT = 'missing', 'same', 'different'
COMPARE_LEVELS = [MISSING, SAME, DIFFERENT]
_num_re = re.compile(r'\d+')


def _norm(x):
    """Normalize one scalar to a lowercased, stripped python str, or None if empty/NA."""
    if x is None or (np.isscalar(x) and pd.isna(x)):
        return None
    s = str(x).strip().lower()
    return s if s and s not in ('nan', 'none') else None


def _norm_series(s: pd.Series) -> pd.Series:
    return s.map(_norm)


def _parse_set(value) -> set:
    """Parse a set-valued cell (phones) into a set of strings. Handles the native
    array/list Parquet form and the legacy stringified-set form."""
    if isinstance(value, (set, frozenset, list, tuple, np.ndarray)):
        return {str(p).strip() for p in value if str(p).strip()}
    if pd.isna(value) or not isinstance(value, str):
        return set()
    v = value.strip()
    if v in ('', 'nan', 'None', 'set()', '{}', '[]'):
        return set()
    try:
        parsed = ast.literal_eval(v)
        if isinstance(parsed, (set, list, tuple)):
            return {str(p).strip() for p in parsed if str(p).strip()}
    except (ValueError, SyntaxError, TypeError):
        pass
    cleaned = v.strip('{}[]').replace("'", '').replace('"', '')
    return {p.strip() for p in cleaned.split(',') if p.strip()}

### 5.1 v1 categorical comparisons (`same` / `different` / `missing`)
Kept unchanged from v1, for all eight fields.

In [ ]:
def compare_field(df: pd.DataFrame, name: str, col: str) -> pd.Categorical:
    """3-level (missing/same/different) comparison. Set fields (SET_FIELDS) are 'same'
    when the two records share >= 1 value; scalar fields on normalized exact equality."""
    if name in SET_FIELDS:
        a = df[f'{col}_A'].apply(_parse_set)
        b = df[f'{col}_B'].apply(_parse_set)
        missing = (a.map(len) == 0) | (b.map(len) == 0)
        same = pd.Series([bool(x & y) for x, y in zip(a, b)], index=df.index) & ~missing
    else:
        a = _norm_series(df[f'{col}_A'])
        b = _norm_series(df[f'{col}_B'])
        missing = a.isna() | b.isna()
        same = (a == b) & ~missing
    out = np.where(missing, MISSING, np.where(same, SAME, DIFFERENT))
    return pd.Categorical(out, categories=COMPARE_LEVELS)


feat = pd.DataFrame(index=pairs.index)
for name in SELECTED_FIELDS:
    feat[f'cmp_{name}'] = compare_field(pairs, name, AVAILABLE_FIELDS[name])

cat_features = [f'cmp_{n}' for n in SELECTED_FIELDS]
num_features = []
print('v1 categorical features:', cat_features)

### 5.2 String similarity — Jaro-Winkler + normalized Levenshtein

In [ ]:
def _jw(x, y):
    return jellyfish.jaro_winkler_similarity(x, y) if (x and y) else np.nan


def _lev_sim(x, y):
    if not x or not y:
        return np.nan
    m = max(len(x), len(y))
    return 1.0 - jellyfish.levenshtein_distance(x, y) / m if m else np.nan


def sim_column(col, fn):
    a = _norm_series(pairs[f'{col}_A'])
    b = _norm_series(pairs[f'{col}_B'])
    return pd.Series([fn(x, y) for x, y in zip(a, b)], index=pairs.index, dtype='float')


JW_FIELDS = ['first_name', 'last_name', 'middle_name', 'email', 'address1']
LEV_FIELDS = ['email', 'address1']

for name in JW_FIELDS:
    col = AVAILABLE_FIELDS[name]
    feat[f'sim_jw_{name}'] = sim_column(col, _jw)
    num_features.append(f'sim_jw_{name}')

for name in LEV_FIELDS:
    col = AVAILABLE_FIELDS[name]
    feat[f'sim_lev_{name}'] = sim_column(col, _lev_sim)
    num_features.append(f'sim_lev_{name}')

print('similarity features:', [c for c in num_features])

### 5.3 Street number, phonetics, name-swap, DOB deltas, SSN digits, address tokens

In [ ]:
# --- Street number: first numeric token of the address, compared exactly. ---
def _street_num(x):
    s = _norm(x)
    if not s:
        return None
    m = _num_re.search(s)
    return m.group() if m else None


sn_a = pairs['AddressLine1_clean_A'].map(_street_num)
sn_b = pairs['AddressLine1_clean_B'].map(_street_num)
sn_missing = sn_a.isna() | sn_b.isna()
sn_same = (sn_a == sn_b) & ~sn_missing
feat['cmp_street_num'] = pd.Categorical(
    np.where(sn_missing, MISSING, np.where(sn_same, SAME, DIFFERENT)), categories=COMPARE_LEVELS)
cat_features.append('cmp_street_num')


# --- Phonetic name agreement (Metaphone): catches sound-alike spellings. ---
def _metaphone(x):
    s = _norm(x)
    if not s:
        return None
    code = jellyfish.metaphone(s)
    return code or None


def phonetic_cmp(col):
    a = pairs[f'{col}_A'].map(_metaphone)
    b = pairs[f'{col}_B'].map(_metaphone)
    missing = a.isna() | b.isna()
    same = (a == b) & ~missing
    return pd.Categorical(np.where(missing, MISSING, np.where(same, SAME, DIFFERENT)), categories=COMPARE_LEVELS)


feat['phon_first_match'] = phonetic_cmp('FirstNM_clean')
feat['phon_last_match'] = phonetic_cmp('LastNM_clean')
cat_features += ['phon_first_match', 'phon_last_match']

In [ ]:
# --- Name-swap: high only when first_A~last_B AND last_A~first_B (min of the two JWs). ---
fa = _norm_series(pairs['FirstNM_clean_A']); la = _norm_series(pairs['LastNM_clean_A'])
fb = _norm_series(pairs['FirstNM_clean_B']); lb = _norm_series(pairs['LastNM_clean_B'])
cross1 = [_jw(x, y) for x, y in zip(fa, lb)]   # first_A vs last_B
cross2 = [_jw(x, y) for x, y in zip(la, fb)]   # last_A  vs first_B
feat['name_swap_score'] = pd.Series(
    [np.nan if (p != p or q != q) else min(p, q) for p, q in zip(cross1, cross2)],
    index=pairs.index, dtype='float')
num_features.append('name_swap_score')

# --- Birth-date deltas: absolute day gap + month/day transposition flag. ---
dob_a = pd.to_datetime(pairs['BirthDT_clean_A'], errors='coerce')
dob_b = pd.to_datetime(pairs['BirthDT_clean_B'], errors='coerce')
feat['dob_abs_days'] = (dob_a - dob_b).abs().dt.days.astype('float')
both_dob = dob_a.notna() & dob_b.notna()
md_swap = (both_dob & (dob_a.dt.year == dob_b.dt.year)
           & (dob_a.dt.month == dob_b.dt.day) & (dob_a.dt.day == dob_b.dt.month)
           & (dob_a.dt.month != dob_a.dt.day))
feat['dob_md_swap'] = np.where(both_dob, md_swap.astype(float), np.nan)
num_features += ['dob_abs_days', 'dob_md_swap']

In [ ]:
# --- SSN digit-level agreement: fraction of position-wise matching digits (full SSN). ---
def _ssn_frac(x, y):
    x, y = _norm(x), _norm(y)
    if not x or not y:
        return np.nan
    n = min(len(x), len(y))
    if n == 0:
        return np.nan
    matches = sum(1 for i in range(n) if x[i] == y[i])
    return matches / max(len(x), len(y))


feat['ssn_digit_frac'] = pd.Series(
    [_ssn_frac(x, y) for x, y in zip(pairs['SSN_clean_A'], pairs['SSN_clean_B'])],
    index=pairs.index, dtype='float')
num_features.append('ssn_digit_frac')


# --- Address token overlap (Jaccard): complements street-number on the street name. ---
def _tok_jaccard(x, y):
    x, y = _norm(x), _norm(y)
    if not x or not y:
        return np.nan
    sx, sy = set(x.split()), set(y.split())
    if not sx or not sy:
        return np.nan
    return len(sx & sy) / len(sx | sy)


feat['addr_token_jaccard'] = pd.Series(
    [_tok_jaccard(x, y) for x, y in zip(pairs['AddressLine1_clean_A'], pairs['AddressLine1_clean_B'])],
    index=pairs.index, dtype='float')
num_features.append('addr_token_jaccard')

### 5.4 Aggregate evidence features
Ambiguity tends to arise from sparse or conflicting evidence, so we summarize the per-field comparisons directly.

In [ ]:
base_cmp = [f'cmp_{n}' for n in SELECTED_FIELDS]
cmp_vals = feat[base_cmp].apply(lambda s: s.astype('object'))

feat['n_same'] = (cmp_vals == SAME).sum(axis=1).astype('float')
feat['n_different'] = (cmp_vals == DIFFERENT).sum(axis=1).astype('float')
feat['n_missing'] = (cmp_vals == MISSING).sum(axis=1).astype('float')

strong_ids = [f'cmp_{n}' for n in ['ssn', 'email', 'phones'] if f'cmp_{n}' in feat]
feat['n_strong_id_agree'] = (feat[strong_ids].apply(lambda s: s.astype('object')) == SAME).sum(axis=1).astype('float')

num_features += ['n_same', 'n_different', 'n_missing', 'n_strong_id_agree']

# Ensure categorical columns carry category dtype for LightGBM.
for c in cat_features:
    feat[c] = feat[c].astype('category')

FEATURE_COLS = cat_features + num_features
print(f'{len(cat_features)} categorical + {len(num_features)} numeric = {len(FEATURE_COLS)} features')
print('\ncategorical:', cat_features)
print('\nnumeric    :', num_features)
feat[FEATURE_COLS].head()

### 5.5 Feature ↔ target signal check

In [ ]:
# Numeric features: mean by class (0=match, 1=ambiguous) and missing rate.
y_full = pairs['target_ambiguous']
rows = []
for c in num_features:
    rows.append({
        'feature': c,
        'mean_match': feat.loc[y_full == 0, c].mean(),
        'mean_ambiguous': feat.loc[y_full == 1, c].mean(),
        'pct_missing': feat[c].isna().mean(),
    })
num_signal = pd.DataFrame(rows).round(3)
print('Numeric features — mean by class:')
num_signal

In [ ]:
# Categorical features: share of ambiguous pairs within each comparison outcome.
for c in cat_features:
    ct = pairs.groupby(feat[c].astype('object'), observed=True)['target_ambiguous'].mean().round(3)
    print(f'{c:20s} P(ambiguous | outcome): ' + '  '.join(f'{k}={v}' for k, v in ct.items()))

## 6. Train / validation / test split

Random stratified split (60/20/20) on the target — **identical to v1** (same `RANDOM_STATE`),
so v2's gains are attributable to features, not to a different split.

> Caveat (as in v1): a random *pair* split can place two pairs sharing a record on opposite
> sides, making the test estimate mildly optimistic. A PATID-grouped split is a future step.

In [ ]:
X = feat[FEATURE_COLS]
y = pairs['target_ambiguous'].astype(int)

idx = np.arange(len(X))
idx_trainval, idx_test = train_test_split(idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y.iloc[idx_trainval])

X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
X_val,   y_val   = X.iloc[idx_val],   y.iloc[idx_val]
X_test,  y_test  = X.iloc[idx_test],  y.iloc[idx_test]

for name, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{name:5s}: n={len(yy):5d}  ambiguous={yy.sum():5d}  amb_rate={yy.mean():.3f}')

## 7. Model training

Same LightGBM configuration as v1 (only the feature set differs). Categorical features are
declared explicitly; numeric features carry `NaN` for missing and LightGBM handles them natively.

In [ ]:
params = dict(
    objective='binary',
    n_estimators=1000,       # upper bound; early stopping picks the real count
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)
print('best iteration:', model.best_iteration_)
print('best val AUC  :', round(model.best_score_['valid_0']['auc'], 4))

## 8. Model evaluation (held-out test set)

Positive class = **ambiguous** (route to human review). As in v1 the threshold favors recall —
missing a hard case is the costly error. Tune `DECISION_THRESHOLD` to the review-queue capacity.

In [ ]:
DECISION_THRESHOLD = 0.3

proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= DECISION_THRESHOLD).astype(int)

print('Classification report (test):\n')
print(classification_report(y_test, pred_test, target_names=['confident match (0)', 'ambiguous (1)'], digits=3))
print(f'ROC AUC : {roc_auc_score(y_test, proba_test):.4f}')
print(f'PR  AUC : {average_precision_score(y_test, proba_test):.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
labels = ['match', 'ambiguous']
cm = confusion_matrix(y_test, pred_test)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax[0], colorbar=False, cmap='Blues')
ax[0].set_title('Confusion matrix (counts, test)')
cm_norm = confusion_matrix(y_test, pred_test, normalize='true')
ConfusionMatrixDisplay(cm_norm, display_labels=labels).plot(ax=ax[1], colorbar=False, cmap='Blues', values_format='.3f')
ax[1].set_title('Confusion matrix (row-normalized)')
plt.tight_layout()
plt.show()

In [ ]:
# ROC and Precision-Recall curves.
fpr, tpr, _ = roc_curve(y_test, proba_test)
prec, rec, _ = precision_recall_curve(y_test, proba_test)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, proba_test):.3f}')
ax[0].plot([0, 1], [0, 1], '--', color='grey', linewidth=1)
ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curve')
ax[0].legend(loc='lower right')
ax[1].plot(rec, prec, label=f'AP = {average_precision_score(y_test, proba_test):.3f}')
ax[1].axhline(y_test.mean(), ls='--', color='grey', linewidth=1, label=f'baseline = {y_test.mean():.3f}')
ax[1].set(xlabel='Recall', ylabel='Precision', title='Precision-Recall curve')
ax[1].legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (gain).
imp = pd.DataFrame({
    'feature': model.feature_name_,
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'split': model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

fig, ax = plt.subplots(figsize=(7, 0.34 * len(imp) + 1.5))
ax.barh(imp['feature'][::-1], imp['gain'][::-1], color='#4C72B0')
ax.set(title='Feature importance (gain) — v2', xlabel='total gain')
plt.tight_layout()
plt.show()
imp

In [ ]:
# Predicted-probability distribution by true class.
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 31)
ax.hist(proba_test[y_test == 0], bins=bins, alpha=0.6, label='confident match (0)', color='#4C72B0')
ax.hist(proba_test[y_test == 1], bins=bins, alpha=0.6, label='ambiguous (1)', color='#C44E52')
ax.axvline(DECISION_THRESHOLD, ls='--', color='black', linewidth=1, label='threshold')
ax.set(xlabel='predicted P(ambiguous)', ylabel='count', title='Score distribution by true class (test)')
ax.legend()
plt.tight_layout()
plt.show()

## 9. v1-vs-v2 ablation — did the new features help?

Retrain the **same LightGBM config** on only the v1 feature set (the eight `cmp_<field>`
categoricals) on the **same split**, and compare threshold-free metrics against the full v2
model. Any gap is the lift from v2's continuous / derived / aggregate features.

In [ ]:
v1_cols = [f'cmp_{n}' for n in SELECTED_FIELDS]

m_v1 = lgb.LGBMClassifier(**params)
m_v1.fit(
    X_train[v1_cols], y_train,
    eval_set=[(X_val[v1_cols], y_val)],
    eval_metric='auc',
    categorical_feature=v1_cols,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)],
)
proba_v1 = m_v1.predict_proba(X_test[v1_cols])[:, 1]

comparison = pd.DataFrame({
    'model': ['v1 (8 categoricals)', 'v2 (full feature set)'],
    'n_features': [len(v1_cols), len(FEATURE_COLS)],
    'ROC_AUC': [roc_auc_score(y_test, proba_v1), roc_auc_score(y_test, proba_test)],
    'PR_AUC': [average_precision_score(y_test, proba_v1), average_precision_score(y_test, proba_test)],
}).round(4)
print('Held-out test comparison (same split, same config):\n')
comparison

In [ ]:
# ROC curves overlaid: v1 vs v2.
fpr1, tpr1, _ = roc_curve(y_test, proba_v1)
fpr2, tpr2, _ = roc_curve(y_test, proba_test)
plt.figure(figsize=(6, 5))
plt.plot(fpr1, tpr1, label=f'v1  AUC={roc_auc_score(y_test, proba_v1):.3f}', color='#8C8C8C')
plt.plot(fpr2, tpr2, label=f'v2  AUC={roc_auc_score(y_test, proba_test):.3f}', color='#4C72B0')
plt.plot([0, 1], [0, 1], '--', color='lightgrey', linewidth=1)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.title('ROC: v1 vs v2 (test)'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

### Notes for the next iteration

- **Leakage-safe split** — move to a PATID-grouped split for a stricter generalization estimate.
- **Nickname / initial logic** — map `J`↔`John`, `Bob`↔`Robert`; treat middle-name present-vs-absent as neutral.
- **Threshold tuning** — set the operating point from the review queue's capacity.
- **Hyperparameter search** — now that the feature set is richer, tune `num_leaves`, `learning_rate`, regularization.
- **Feature pruning** — drop low-gain features (watch the importance table) to keep the model lean.